# Ionomer — Noble-Metal Nanoparticle Detection & Segmentation (Merged)



**Pipeline**: load DM4 -> separate sample/vacuum + normalize -> CLAHE local enhancement -> region-based LoG candidate centers (coarse + residual passes) -> SAM segmentation in three prompt modes (point / rescue / box) -> union -> statistics + export.



## 0. Environment & dependencies

In [ ]:
# Install dependencies (first run; skip this cell if your local env already has them)
!pip install -q "hyperspy[all]" scikit-image scikit-learn scipy matplotlib pandas seaborn numba
!pip install -q segment-anything

In [ ]:
# Download SAM weights (vit_b). More robust than wget; works on Windows / Mac / Linux / Colab.
import os, urllib.request

SAM_CKPT = "sam_vit_b_01ec64.pth"
SAM_URL  = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not os.path.exists(SAM_CKPT):
    print("Downloading SAM checkpoint ...")
    urllib.request.urlretrieve(SAM_URL, SAM_CKPT)
print("SAM checkpoint ready:", SAM_CKPT,
      f"({os.path.getsize(SAM_CKPT)/1e6:.0f} MB)")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

import hyperspy.api as hs
import torch

from skimage import morphology, measure, segmentation, exposure
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries, watershed
from skimage.feature import blob_log
from scipy import ndimage
from scipy.ndimage import gaussian_filter
from scipy.stats import norm

from segment_anything import sam_model_registry, SamPredictor

import gc, glob, warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:


DM4_FILE = os.path.join(DATA_DIR, '145536.dm4')
print("IN_COLAB =", IN_COLAB)
print("DM4_FILE =", DM4_FILE)


## 1. Preprocessing & candidate-center detection (former Part 1)

Separate sample from vacuum, normalize against the sample region, apply CLAHE, then run region-wise (thick/thin) LoG detection of candidate particle centers, with a second residual pass to recover misses. Produces `filtered_blobs`.

### 1.0 Load DM4

In [ ]:
s = hs.load(DM4_FILE)

print("File info:")
print(f"  type  : {type(s)}")
print(f"  shape : {s.data.shape}")
print(f"  dtype : {s.data.dtype}")

### 1.1 Diagnostic: sample / vacuum histogram

In [ ]:
# ============================================================
# Diagnostic: histogram of sample region only
# ============================================================
img_raw = s
# Step 1: separate vacuum from sample
# vacuum pixels have intensity very close to 0
vacuum_thresh_raw = np.percentile(img_raw.data[img_raw.data > 0], 5)
sample_mask = img_raw.data > vacuum_thresh_raw

print(f"Total pixels       : {img_raw.data.size}")
print(f"Vacuum pixels      : {(~sample_mask).sum()} "
      f"({(~sample_mask).sum()/img_raw.data.size*100:.1f}%)")
print(f"Sample pixels      : {sample_mask.sum()} "
      f"({sample_mask.sum()/img_raw.data.size*100:.1f}%)")

# Step 2: histogram of sample-only pixels
sample_pixels = img_raw.data[sample_mask]

print(f"\nSample pixel stats:")
print(f"  min    : {sample_pixels.min():.1f}")
print(f"  max    : {sample_pixels.max():.1f}")
print(f"  mean   : {sample_pixels.mean():.1f}")
print(f"  median : {np.median(sample_pixels):.1f}")
print(f"  p25    : {np.percentile(sample_pixels, 25):.1f}")
print(f"  p75    : {np.percentile(sample_pixels, 75):.1f}")
print(f"  p95    : {np.percentile(sample_pixels, 95):.1f}")
print(f"  p99    : {np.percentile(sample_pixels, 99):.1f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full histogram (all pixels)
axes[0].hist(img_raw.data.flatten(), bins=300, density=True,
             color='steelblue', alpha=0.7)
axes[0].set_title('All pixels (vacuum dominates)')
axes[0].set_xlabel('Raw Intensity')
axes[0].set_ylabel('Probability Density')

# Sample-only histogram
axes[1].hist(sample_pixels, bins=300, density=True,
             color='orangered', alpha=0.7)
axes[1].set_title('Sample pixels only\n(vacuum excluded)')
axes[1].set_xlabel('Raw Intensity')
axes[1].set_ylabel('Probability Density')
# Mark percentiles
for p, c in [(50,'blue'), (90,'green'), (99,'red')]:
    val = np.percentile(sample_pixels, p)
    axes[1].axvline(val, color=c, lw=1.5, linestyle='--',
                    label=f'p{p}={val:.0f}')
axes[1].legend(fontsize=9)

# Sample mask visualization
axes[2].imshow(sample_mask, cmap='gray')
axes[2].set_title('Sample mask\n(white = sample, black = vacuum)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('diagnostic_sample_mask.png', dpi=150,
            bbox_inches='tight')
plt.show()

### 1.2 Normalization (sample-based)

In [ ]:
# ============================================================
# Correct normalization: based on sample region only
# ============================================================

# Build sample mask (exclude vacuum)
vacuum_thresh_raw = np.percentile(img_raw.data[img_raw.data > 0], 5)
sample_mask_norm  = img_raw.data > vacuum_thresh_raw

# Compute percentiles from sample pixels only
sample_pixels = img_raw.data[sample_mask_norm]
val_low  = np.percentile(sample_pixels, 1)    # dark end of sample
val_high = np.percentile(sample_pixels, 99)   # bright end of sample

print(f"Normalization range (sample only): "
      f"[{val_low:.1f}, {val_high:.1f}]")

# Normalize
img = np.clip(img_raw.data, val_low, val_high)
img = (img - val_low) / (val_high - val_low)

# Keep vacuum as 0
img[~sample_mask_norm] = 0.0

print(f"Normalized range: [{img.min():.4f}, {img.max():.4f}]")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Normalized Image\n(sample-based normalization)')
axes[0].axis('off')

# Histogram of sample region after normalization
axes[1].hist(img[sample_mask_norm].flatten(), bins=300,
             density=True, color='steelblue', alpha=0.7)
axes[1].set_title('Normalized Intensity\n(sample pixels only)')
axes[1].set_xlabel('Normalized Intensity')
axes[1].set_ylabel('Probability Density')
axes[1].axvline(img[sample_mask_norm].mean(), color='orange',
                lw=1.5, linestyle='--',
                label=f'mean={img[sample_mask_norm].mean():.3f}')
axes[1].legend()

# Log-scale histogram to see particle tail
axes[2].hist(img[sample_mask_norm].flatten(), bins=300,
             density=True, color='orangered', alpha=0.7,
             log=True)
axes[2].set_title('Normalized Intensity (log scale)\n'
                   'particle tail visible on right')
axes[2].set_xlabel('Normalized Intensity')
axes[2].set_ylabel('Log Probability Density')

plt.tight_layout()
plt.savefig('step0_norm_corrected.png', dpi=150,
            bbox_inches='tight')
plt.show()

### 1.3 CLAHE local contrast enhancement

In [ ]:
# ============================================================
# CLAHE local contrast enhancement
# ============================================================
from skimage import exposure

# clip_limit: controls contrast enhancement strength
#   0.01 = mild,  0.03 = moderate,  0.05 = strong
# kernel_size: size of local region for histogram equalization
#   smaller = more local adaptation
#   larger  = more global
# Rule of thumb: kernel_size ≈ image_size / 8

clip_limit  = 0.03
kernel_size = 64    # adjust if image size is very different from 512px

img_clahe = exposure.equalize_adapthist(
    img,
    clip_limit=clip_limit,
    kernel_size=kernel_size
)

print(f"CLAHE parameters:")
print(f"  clip_limit  : {clip_limit}")
print(f"  kernel_size : {kernel_size}")
print(f"  Output range: [{img_clahe.min():.4f}, {img_clahe.max():.4f}]")

# ── Visualize ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Images
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Normalized Image (before CLAHE)')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_clahe, cmap='gray')
axes[0, 1].set_title(f'After CLAHE\n'
                      f'(clip={clip_limit}, kernel={kernel_size})')
axes[0, 1].axis('off')

# Difference map: what CLAHE changed
diff = img_clahe - img
axes[0, 2].imshow(diff, cmap='RdBu_r',
                   vmin=-diff.std()*3, vmax=diff.std()*3)
axes[0, 2].set_title('CLAHE - Original\n'
                      '(blue=darkened, red=brightened)')
axes[0, 2].axis('off')
plt.colorbar(axes[0, 2].images[0], ax=axes[0, 2], fraction=0.046)

# Histograms (sample region only)
sample_before = img[sample_mask_norm].flatten()
sample_after  = img_clahe[sample_mask_norm].flatten()

axes[1, 0].hist(sample_before, bins=200, density=True,
                color='steelblue', alpha=0.7,
                label='Before CLAHE')
axes[1, 0].hist(sample_after, bins=200, density=True,
                color='orangered', alpha=0.5,
                label='After CLAHE')
axes[1, 0].set_xlabel('Intensity')
axes[1, 0].set_ylabel('Probability Density')
axes[1, 0].set_title('Intensity distribution comparison')
axes[1, 0].legend()

# Log scale
axes[1, 1].hist(sample_before, bins=200, density=True,
                color='steelblue', alpha=0.7, log=True,
                label='Before CLAHE')
axes[1, 1].hist(sample_after, bins=200, density=True,
                color='orangered', alpha=0.5, log=True,
                label='After CLAHE')
axes[1, 1].set_xlabel('Intensity')
axes[1, 1].set_ylabel('Log Probability Density')
axes[1, 1].set_title('Log scale comparison\n'
                      '(CLAHE should spread the distribution)')
axes[1, 1].legend()

# Zoomed comparison: thick region (center) vs thin region (edge)
# Pick a small patch from thick region and thin region
h, w = img.shape
cy, cx = h // 2, w // 2          # center (likely thick region)
patch_size = 80

# Thick region patch
y1, y2 = cy - patch_size, cy + patch_size
x1, x2 = cx - patch_size, cx + patch_size
y1, y2 = max(0, y1), min(h, y2)
x1, x2 = max(0, x1), min(w, x2)

patch_before = img[y1:y2, x1:x2]
patch_after  = img_clahe[y1:y2, x1:x2]

# Show side-by-side zoom
zoom_compare = np.concatenate([patch_before, patch_after], axis=1)
axes[1, 2].imshow(zoom_compare, cmap='gray')
axes[1, 2].set_title('Zoom: center region\n'
                      'Left=before, Right=after CLAHE\n'
                      '(particles should be more distinct on right)')
axes[1, 2].axis('off')
axes[1, 2].axvline(patch_size * 2, color='yellow',
                    lw=1, alpha=0.8)

plt.tight_layout()
plt.savefig('step_clahe.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Quick check: does CLAHE help distinguish particles? ───────
print("\nCheck:")
print("  1. Is the CLAHE image visually clearer than the original?")
print("  2. Are particles in the thick (bright) region more visible?")
print("  3. Is the histogram more spread out (less skewed)?")
print("\nIf particles look over-enhanced or grainy:")
print(f"  -> decrease clip_limit (currently {clip_limit})")
print(f"     try: 0.01 or 0.02")
print("\nIf thick/thin regions still look very different:")
print(f"  -> decrease kernel_size (currently {kernel_size})")
print(f"     try: 32 or 48")

### 1.4 Region-based LoG detection (thick / thin split + local intensity filter)

In [ ]:
# ============================================================
# Region-based LoG detection
# Different threshold for thick vs thin regions
# ============================================================
from skimage.feature import blob_log
from scipy.ndimage import gaussian_filter

# ── Step 1: Separate thick and thin regions ──────────────────
# Use heavily smoothed image to find slow-varying thickness map
thickness_map = gaussian_filter(img, sigma=40)

# Threshold the thickness map to define thick/thin regions
thick_thresh = np.percentile(
    thickness_map[sample_mask_norm], 70)  # top 30% = thick

thick_mask = (thickness_map > thick_thresh) & sample_mask_norm
thin_mask  = (thickness_map <= thick_thresh) & sample_mask_norm

print(f"Thick region: {thick_mask.sum()/sample_mask_norm.sum()*100:.1f}% of sample")
print(f"Thin  region: {thin_mask.sum()/sample_mask_norm.sum()*100:.1f}% of sample")

# Visualize the region separation
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(thickness_map, cmap='hot')
axes[1].set_title('Thickness map (smoothed, sigma=40)\n'
                   'bright = thick regions')
axes[1].axis('off')
plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046)

region_vis = np.zeros((*img.shape, 3))
region_vis[thick_mask] = [1, 0.3, 0.3]   # red = thick
region_vis[thin_mask]  = [0.3, 0.3, 1]   # blue = thin
axes[2].imshow(img, cmap='gray', alpha=0.6)
axes[2].imshow(region_vis, alpha=0.4)
axes[2].set_title('Region map\nred=thick, blue=thin')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('step_regions.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Step 2: LoG on each region separately ───────────────────
# Thin region: higher threshold (good contrast, don't need low threshold)
# Thick region: lower threshold (poor contrast, need to be more sensitive)

def detect_blobs_in_region(img_clahe, region_mask,
                            threshold, min_sigma=2, max_sigma=20,
                            num_sigma=15, overlap=0.4):
    """Run LoG and keep only detections inside region_mask."""
    blobs = blob_log(img_clahe,
                     min_sigma=min_sigma,
                     max_sigma=max_sigma,
                     num_sigma=num_sigma,
                     threshold=threshold,
                     overlap=overlap)
    if len(blobs) == 0:
        return np.array([]).reshape(0, 3)
    valid = [b for b in blobs
             if region_mask[int(b[0]), int(b[1])]]
    return np.array(valid) if valid else np.array([]).reshape(0, 3)

# Detect separately
blobs_thin  = detect_blobs_in_region(
    img_clahe, thin_mask,
    threshold=0.18,   # stricter for thin regions (good contrast)
    min_sigma=2, max_sigma=20)

blobs_thick = detect_blobs_in_region(
    img_clahe, thick_mask,
    threshold=0.12,   # more sensitive for thick regions (poor contrast)
    min_sigma=2, max_sigma=25)

print(f"\nThin  region blobs: {len(blobs_thin)}")
print(f"Thick region blobs: {len(blobs_thick)}")

# Combine
if len(blobs_thin) > 0 and len(blobs_thick) > 0:
    all_blobs = np.vstack([blobs_thin, blobs_thick])
elif len(blobs_thin) > 0:
    all_blobs = blobs_thin
elif len(blobs_thick) > 0:
    all_blobs = blobs_thick
else:
    all_blobs = np.array([]).reshape(0, 3)

print(f"Total combined    : {len(all_blobs)}")

# ── Step 3: Intensity filter per region ─────────────────────
local_radius = 5
filtered_blobs = []

for blob in all_blobs:
    y, x, sigma = blob
    y_int, x_int = int(y), int(x)
    y1 = max(0, y_int - local_radius)
    y2 = min(img.shape[0], y_int + local_radius)
    x1 = max(0, x_int - local_radius)
    x2 = min(img.shape[1], x_int + local_radius)
    patch = img[y1:y2, x1:x2]

    # Use local background as reference
    # particle must be brighter than its local neighborhood
    local_bg = thickness_map[y_int, x_int]
    if patch.mean() > local_bg * 1.3:  # at least 30% above local bg
        filtered_blobs.append(blob)

filtered_blobs = np.array(filtered_blobs)
print(f"After local filter: {len(filtered_blobs)}")

# ── Visualize final result ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Full image
axes[0, 0].imshow(img_clahe, cmap='gray')
axes[0, 0].set_title(f'All detections: {len(all_blobs)}')
axes[0, 0].axis('off')
if len(all_blobs) > 0:
    axes[0, 0].scatter(all_blobs[:, 1], all_blobs[:, 0],
                       s=6, c='red', marker='+',
                       linewidths=0.7, alpha=0.6)

axes[0, 1].imshow(img_clahe, cmap='gray')
axes[0, 1].set_title(f'After local filter: {len(filtered_blobs)}')
axes[0, 1].axis('off')
if len(filtered_blobs) > 0:
    axes[0, 1].scatter(filtered_blobs[:, 1], filtered_blobs[:, 0],
                       s=6, c='lime', marker='+',
                       linewidths=0.7, alpha=0.8)

# Size distribution
if len(filtered_blobs) > 0:
    radii = filtered_blobs[:, 2] * np.sqrt(2)
    axes[0, 2].hist(radii, bins=30, color='steelblue',
                    edgecolor='white')
    axes[0, 2].set_xlabel('Detected Radius (px)')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].set_title('Size distribution after filtering')
    axes[0, 2].axvline(np.median(radii), color='red',
                       linestyle='--',
                       label=f'median={np.median(radii):.1f}px')
    axes[0, 2].legend()

# Zoom into 3 zones
h, w = img.shape
zones = {
    'Upper left (thin)':  (0,      h//3,   0,      w//2),
    'Center (thick)':     (h//3,   2*h//3, w//4,   3*w//4),
    'Lower right (thin)': (2*h//3, h,      w//2,   w),
}

for ax, (name, (y1, y2, x1, x2)) in zip(axes[1], zones.items()):
    ax.imshow(img_clahe[y1:y2, x1:x2], cmap='gray')
    ax.set_title(f'Zoom: {name}')
    ax.axis('off')
    if len(filtered_blobs) > 0:
        in_zone = filtered_blobs[
            (filtered_blobs[:, 0] >= y1) &
            (filtered_blobs[:, 0] <  y2) &
            (filtered_blobs[:, 1] >= x1) &
            (filtered_blobs[:, 1] <  x2)
        ]
        ax.scatter(in_zone[:, 1] - x1,
                   in_zone[:, 0] - y1,
                   s=20, c='lime', marker='+',
                   linewidths=1.2, alpha=0.9)

plt.tight_layout()
plt.savefig('step_region_detection.png', dpi=150,
            bbox_inches='tight')
plt.show()

print(f"\nAdjust if needed:")
print(f"  Too many in thin region  -> raise thin threshold: 0.20 or 0.22")
print(f"  Too few  in thick region -> lower thick threshold: 0.10")
print(f"  Too many overall         -> raise local filter: 1.3 -> 1.5")
print(f"  Too few  overall         -> lower local filter: 1.3 -> 1.1")

### 1.5 Residual pass (second round, recover missed particles)

In [ ]:
# ============================================================
# Residual detection: find missed particles
# ============================================================

# Step 1: Build a mask of already-detected regions
detected_mask = np.zeros(img.shape, dtype=bool)
for blob in filtered_blobs:
    y, x, sigma = blob
    y_int, x_int = int(y), int(x)
    r = int(sigma * np.sqrt(2)) + 2
    rr, cc = np.ogrid[-r:r+1, -r:r+1]
    circle = rr**2 + cc**2 <= r**2
    y1 = max(0, y_int - r)
    y2 = min(img.shape[0], y_int + r + 1)
    x1 = max(0, x_int - r)
    x2 = min(img.shape[1], x_int + r + 1)
    cy1 = r - (y_int - y1)
    cy2 = cy1 + (y2 - y1)
    cx1 = r - (x_int - x1)
    cx2 = cx1 + (x2 - x1)
    detected_mask[y1:y2, x1:x2] |= circle[cy1:cy2, cx1:cx2]

# Step 2: Create residual image
# Replace detected regions with local background estimate
img_residual = img_clahe.copy()
img_residual[detected_mask] = thickness_map[detected_mask]

# Step 3: Run LoG again on residual image
blobs_residual = blob_log(
    img_residual,
    min_sigma=2,
    max_sigma=20,
    num_sigma=15,
    threshold=0.10,    # lower threshold to catch weaker missed particles
    overlap=0.3
)

# Keep only inside sample mask and outside already detected regions
if len(blobs_residual) > 0:
    valid_residual = []
    for b in blobs_residual:
        y, x, sigma = b
        y_int, x_int = int(y), int(x)
        if (sample_mask_norm[y_int, x_int] and
                not detected_mask[y_int, x_int]):
            valid_residual.append(b)
    blobs_residual = (np.array(valid_residual)
                      if valid_residual
                      else np.array([]).reshape(0, 3))

# Apply local intensity filter to residual detections
extra_blobs = []
for blob in blobs_residual:
    y, x, sigma = blob
    y_int, x_int = int(y), int(x)
    y1 = max(0, y_int - local_radius)
    y2 = min(img.shape[0], y_int + local_radius)
    x1 = max(0, x_int - local_radius)
    x2 = min(img.shape[1], x_int + local_radius)
    patch_mean = img[y1:y2, x1:x2].mean()
    local_bg   = thickness_map[y_int, x_int]
    if patch_mean > local_bg * 1.2:   # slightly relaxed threshold
        extra_blobs.append(blob)

extra_blobs = (np.array(extra_blobs)
               if extra_blobs
               else np.array([]).reshape(0, 3))

print(f"Round 1 detections : {len(filtered_blobs)}")
print(f"Round 2 (residual) : {len(extra_blobs)}")

# Combine all detections
if len(extra_blobs) > 0:
    all_final_blobs = np.vstack([filtered_blobs, extra_blobs])
else:
    all_final_blobs = filtered_blobs

print(f"Total combined     : {len(all_final_blobs)}")

# ── Visualize ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 8))

axes[0].imshow(img_clahe, cmap='gray')
axes[0].set_title(f'Round 1 only: {len(filtered_blobs)}')
axes[0].axis('off')
axes[0].scatter(filtered_blobs[:, 1], filtered_blobs[:, 0],
                s=8, c='lime', marker='+',
                linewidths=0.8, alpha=0.8)

axes[1].imshow(img_clahe, cmap='gray')
axes[1].set_title(f'Round 2 additions: {len(extra_blobs)}\n'
                   '(cyan = newly found)')
axes[1].axis('off')
axes[1].scatter(filtered_blobs[:, 1], filtered_blobs[:, 0],
                s=8, c='lime', marker='+',
                linewidths=0.8, alpha=0.6,
                label=f'Round 1: {len(filtered_blobs)}')
if len(extra_blobs) > 0:
    axes[1].scatter(extra_blobs[:, 1], extra_blobs[:, 0],
                    s=12, c='cyan', marker='+',
                    linewidths=1.2, alpha=0.9,
                    label=f'Round 2: {len(extra_blobs)}')
axes[1].legend(loc='upper right', fontsize=9)

axes[2].imshow(img_clahe, cmap='gray')
axes[2].set_title(f'All final: {len(all_final_blobs)}')
axes[2].axis('off')
axes[2].scatter(all_final_blobs[:, 1], all_final_blobs[:, 0],
                s=7, c='lime', marker='+',
                linewidths=0.8, alpha=0.8)

plt.tight_layout()
plt.savefig('step_residual_detection.png', dpi=150,
            bbox_inches='tight')
plt.show()

# Update filtered_blobs for SAM
filtered_blobs = all_final_blobs
print(f"\nFinal blob count for SAM: {len(filtered_blobs)}")
print("Ready to run SAM segmentation.")

### 1.6 (Optional) Save checkpoint

Skip in a single continuous session; used to restore state on a split run.

In [ ]:
# (Optional) save intermediate results for a resumable / split run
np.save('all_final_blobs.npy', filtered_blobs)
np.save('sample_mask_norm.npy', sample_mask_norm)
maybe_download('all_final_blobs.npy')
maybe_download('sample_mask_norm.npy')
print(f"filtered_blobs   : {filtered_blobs.shape}")
print(f"sample_mask_norm : {sample_mask_norm.shape}")

## 2. SAM segmentation

Run SAM on the candidate centers. Three prompt strategies are combined by union:
1. **Chunked point prompts** — memory-friendly, per-batch `gc`, resumable -> `masks_point`
2. **rescue** — recover bright regions the point pass missed, with watershed to split agglomerates -> `masks_rescue`
3. **box prompts** — boxes derived from blob size, run independently -> `masks_box`

### 2.0 Variable-restore guard (works for single or split session)

In [ ]:
# Single continuous session: img / img_clahe / filtered_blobs / sample_mask_norm are already
# in memory, so this cell is a no-op.
# Split run (runtime was restarted): restore blobs/mask from disk and recompute norm + CLAHE.
if 'filtered_blobs' not in globals():
    print("No variables in memory; restoring from disk ...")
    filtered_blobs   = np.load('all_final_blobs.npy')
    sample_mask_norm = np.load('sample_mask_norm.npy')
    s   = hs.load(DM4_FILE)
    raw = s.data
    sp  = raw[sample_mask_norm]
    lo, hi = np.percentile(sp, 1), np.percentile(sp, 99)
    img = np.clip(raw, lo, hi)
    img = (img - lo) / (hi - lo)
    img[~sample_mask_norm] = 0.0
    img_clahe = exposure.equalize_adapthist(img, clip_limit=0.03, kernel_size=64)
    print("Restored. img:", img.shape)
else:
    print("Variables already in memory; skipping reload.")
print("filtered_blobs:", filtered_blobs.shape)

### 2.1 Chunked point-prompt SAM -> `masks_point`

Defaults to `START=0, END=len(filtered_blobs)` (one pass). If GPU memory is tight, set `START/END` to run in several passes, each writing a `masks_chunk_*.npy`, then merge with 2.1b.

In [ ]:
# ============================================================
# Cell 4 v2: Process all blobs in chunks, save after each chunk
# NO watershed for large masks, just strict size filter
# ============================================================
import torch
import gc
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
from skimage import measure
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt
import os

# ── Settings ─────────────────────────────────────────────────
# Change these to process different chunks:
# Chunk 1: START=0,    END=400
# Chunk 2: START=400,  END=800
# Chunk 3: START=800,  END=1309
START = 0
END   = len(filtered_blobs)

# Load existing mask if continuing from previous chunk
CHECKPOINT = None   # set to filename if loading previous chunk
# CHECKPOINT = 'masks_chunk1.npy'  # uncomment for chunk 2 and 3

image_area = img.shape[0] * img.shape[1]
MIN_AREA   = 5
MAX_AREA   = 2000          # strict upper limit, no watershed
HUGE_AREA  = image_area * 0.005   # 0.5% of image = discard

print(f"Image size   : {img.shape}")
print(f"HUGE_AREA    : >{HUGE_AREA:.0f} px²  → discard")
print(f"MAX_AREA     : >{MAX_AREA} px²       → discard")
print(f"Processing   : blobs {START} to {END}")

# ── Load checkpoint if exists ─────────────────────────────────
if CHECKPOINT and os.path.exists(CHECKPOINT):
    all_masks = np.load(CHECKPOINT)
    labels_tmp = measure.label(all_masks)
    print(f"Loaded checkpoint: {labels_tmp.max()} particles")
else:
    all_masks = np.zeros(img.shape, dtype=bool)
    print("Starting from empty mask")

# ── Load SAM ─────────────────────────────────────────────────
sam    = sam_model_registry["vit_b"](
             checkpoint="sam_vit_b_01ec64.pth")
sam.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
sam.to(device)
print(f"SAM on: {device}")

predictor   = SamPredictor(sam)
img_for_sam = (img_clahe * 255).astype(np.uint8)
img_rgb     = np.stack([img_for_sam] * 3, axis=-1)
predictor.set_image(img_rgb)

sample_mean   = img[sample_mask_norm].mean()
valid_count   = 0
skipped_count = 0
BATCH_SIZE    = 50
blobs_run     = filtered_blobs[START:END]

print(f"Blobs to process: {len(blobs_run)}\n")

for batch_start in range(0, len(blobs_run), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(blobs_run))
    batch     = blobs_run[batch_start:batch_end]

    for blob in batch:
        y, x, sigma = blob
        y_int, x_int = int(y), int(x)

        try:
            masks, scores, logits = predictor.predict(
                point_coords=np.array([[x_int, y_int]]),
                point_labels=np.array([1]),
                multimask_output=True
            )
            best_mask = masks[np.argmax(scores)]
            mask_area = best_mask.sum()

            # Strict size filter: discard anything outside range
            if mask_area < MIN_AREA or mask_area > MAX_AREA:
                skipped_count += 1
                continue

            # Discard if covers too much of image
            if mask_area > HUGE_AREA:
                skipped_count += 1
                continue

            # Intensity filter
            mean_int = img[best_mask].mean()
            if mean_int < sample_mean * 0.5:
                skipped_count += 1
                continue

            all_masks   = all_masks | best_mask
            valid_count += 1

        except Exception:
            continue

    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    global_idx = START + batch_end
    print(f"  [{global_idx:4d}/{END}]  "
          f"valid={valid_count}  skipped={skipped_count}")

# ── Save this chunk ───────────────────────────────────────────
chunk_name = f'masks_chunk_{START}_{END}.npy'
np.save(chunk_name, all_masks)
print(f"\nSaved: {chunk_name}")

# ── Quick preview ─────────────────────────────────────────────
labels_tmp = measure.label(all_masks)
print(f"Particles so far: {labels_tmp.max()}")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
overlay = label2rgb(labels_tmp, image=img, bg_label=0, alpha=0.45)
axes[0].imshow(overlay)
axes[0].set_title(f'Chunk result: {labels_tmp.max()} particles')
axes[0].axis('off')

bnd = np.stack([img] * 3, axis=-1)
bnd[find_boundaries(labels_tmp, mode='thick')] = [1, 0.1, 0.1]
axes[1].imshow(bnd)
axes[1].set_title('Boundaries')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(f'preview_chunk_{START}_{END}.png', dpi=150,
            bbox_inches='tight')
plt.show()

# (Colab download handled by maybe_download)
maybe_download(chunk_name)
maybe_download(f'preview_chunk_{START}_{END}.png')
masks_point = all_masks.copy()
print('masks_point particles:', measure.label(masks_point).max())

### 2.1b (Optional) Merge chunks across sessions

Only needed if 2.1 was run in several sessions; keep `False` for a single pass.

In [ ]:
# Merge all masks_chunk_*.npy on disk -> masks_point
MERGE_CHUNKS_FROM_DISK = False

if MERGE_CHUNKS_FROM_DISK:
    chunk_files = sorted(glob.glob('masks_chunk_*.npy'))
    print("Found chunks:", chunk_files)
    masks_point = np.zeros(img.shape, dtype=bool)
    for cf in chunk_files:
        m = np.load(cf)
        masks_point |= m
        print("  ", cf, ":", measure.label(m).max(), "particles")
    print("Merged point-prompt particles:", measure.label(masks_point).max())
else:
    print("Using in-memory masks_point from 2.1:", measure.label(masks_point).max(), "particles")

### 2.2 Point-result statistics & visualization

In [ ]:
labels_sam = measure.label(masks_point)
print("Point-prompt particles:", labels_sam.max())

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
overlay = label2rgb(labels_sam, image=img, bg_label=0, alpha=0.45)
axes[1].imshow(overlay); axes[1].set_title(f'Point prompt: {labels_sam.max()} particles'); axes[1].axis('off')
bnd = np.stack([img] * 3, axis=-1)
bnd[find_boundaries(labels_sam, mode='thick')] = [1, 0.1, 0.1]
axes[2].imshow(bnd); axes[2].set_title('Boundaries'); axes[2].axis('off')
plt.tight_layout(); plt.savefig('sam_point_result.png', dpi=150, bbox_inches='tight'); plt.show()

props = measure.regionprops_table(
    labels_sam, intensity_image=img,
    properties=['label', 'centroid', 'area', 'equivalent_diameter_area',
                'mean_intensity', 'max_intensity', 'perimeter'])
df = pd.DataFrame(props)
df.columns = ['label', 'cy', 'cx', 'area_px2', 'diam_px',
              'mean_int', 'max_int', 'perim']
df['circularity'] = (4 * np.pi * df['area_px2'] / df['perim']**2).round(3)
df_point = df[df['area_px2'] < 5000].copy()
print(f"After size filter: {len(df_point)} particles | "
      f"diam mean={df_point['diam_px'].mean():.1f}px | "
      f"circ={df_point['circularity'].mean():.3f}")
df_point.to_csv('particle_results_point.csv', index=False)
maybe_download('sam_point_result.png')
maybe_download('particle_results_point.csv')

### 2.3 Miss diagnostic (bright-region coverage)

In [ ]:
# ============================================================
# Diagnostic: visualize missed particles
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries

# Current result
labels_sam = measure.label(masks_point)
print(f"Currently detected: {labels_sam.max()} particles")

# Find bright regions not covered by any mask
# Use the normalized image to find bright spots
bright_threshold = img[sample_mask_norm].mean() + \
                   1.2 * img[sample_mask_norm].std()
bright_mask = (img > bright_threshold) & sample_mask_norm

# Regions in bright_mask but NOT in masks_point = missed particles
missed_mask = bright_mask & ~masks_point

print(f"Bright pixel threshold : {bright_threshold:.4f}")
print(f"Bright pixels total    : {bright_mask.sum()}")
print(f"Bright pixels covered  : {(bright_mask & masks_point).sum()}")
print(f"Bright pixels missed   : {missed_mask.sum()}")
coverage = (bright_mask & masks_point).sum() / bright_mask.sum()
print(f"Coverage rate          : {coverage*100:.1f}%")

# Visualize missed regions
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

# Show detected (green) and missed (red) bright regions
vis = np.stack([img] * 3, axis=-1)
vis[masks_point & bright_mask] = [0.2, 0.8, 0.2]   # green = detected
vis[missed_mask]                = [1.0, 0.2, 0.2]   # red   = missed
axes[1].imshow(vis)
axes[1].set_title(f'Green = detected, Red = missed\n'
                   f'Coverage: {coverage*100:.1f}%')
axes[1].axis('off')

# Zoom into upper region where most misses are
h, w = img.shape
axes[2].imshow(vis[0:h//2, 0:w], )
axes[2].set_title('Zoom: upper half\n'
                   'red dots = missed bright particles')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('diagnostic_missed.png', dpi=150, bbox_inches='tight')
plt.show()

# Cluster the missed pixels into individual particles
missed_labels = measure.label(missed_mask)
missed_props  = measure.regionprops(missed_labels)
print(f"\nMissed particle clusters: {missed_labels.max()}")
missed_areas  = [r.area for r in missed_props]
if missed_areas:
    print(f"  Area range: {min(missed_areas)}-{max(missed_areas)} px²")
    print(f"  Mean area : {np.mean(missed_areas):.1f} px²")


### 2.4 rescue: two passes (small particles + agglomerate watershed) -> `masks_rescue`

In [ ]:
# ============================================================
# Rescue Cell: Detect missed particles in two passes
# ============================================================
import torch
import gc
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
from skimage import measure, morphology
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries, watershed
from scipy import ndimage
import matplotlib.pyplot as plt
import pandas as pd

# ── Setup ─────────────────────────────────────────────────────
sam    = sam_model_registry["vit_b"](
             checkpoint="sam_vit_b_01ec64.pth")
sam.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
sam.to(device)

predictor   = SamPredictor(sam)
img_for_sam = (img_clahe * 255).astype(np.uint8)
img_rgb     = np.stack([img_for_sam] * 3, axis=-1)
predictor.set_image(img_rgb)

sample_mean = img[sample_mask_norm].mean()
image_area  = img.shape[0] * img.shape[1]

# Load current best mask
masks_current = masks_point.copy()
labels_before = measure.label(masks_current)
print(f"Starting particles: {labels_before.max()}")

# ── Find missed bright regions ────────────────────────────────
bright_threshold = (img[sample_mask_norm].mean() +
                    1.5 * img[sample_mask_norm].std())
bright_mask  = (img > bright_threshold) & sample_mask_norm
missed_mask  = bright_mask & ~masks_current
missed_labels = measure.label(missed_mask)
missed_props  = measure.regionprops(missed_labels,
                                    intensity_image=img)
print(f"Missed bright clusters: {missed_labels.max()}")

# ============================================================
# Pass 1: Small missed particles (single prompt per cluster)
# ============================================================
print("\n--- Pass 1: Small missed particles ---")
pass1_count = 0

small_clusters = [r for r in missed_props
                  if r.area < 500]   # small isolated misses
print(f"Small clusters to try: {len(small_clusters)}")

for region in small_clusters:
    cy, cx = region.centroid
    y_int, x_int = int(cy), int(cx)

    try:
        masks, scores, logits = predictor.predict(
            point_coords=np.array([[x_int, y_int]]),
            point_labels=np.array([1]),
            multimask_output=True
        )
        best_mask = masks[np.argmax(scores)]
        mask_area = best_mask.sum()

        # Accept only reasonably sized masks
        if mask_area < 5 or mask_area > 2000:
            continue
        if mask_area > image_area * 0.005:
            continue

        mean_int = img[best_mask].mean()
        if mean_int < sample_mean * 0.5:
            continue

        masks_current = masks_current | best_mask
        pass1_count  += 1

    except Exception:
        continue

print(f"Pass 1 added: {pass1_count} particles")

# ============================================================
# Pass 2: Large missed clusters (agglomerates → watershed)
# ============================================================
print("\n--- Pass 2: Large missed clusters (agglomerates) ---")
pass2_count = 0

large_clusters = [r for r in missed_props
                  if r.area >= 500]
print(f"Large clusters to split: {len(large_clusters)}")

for region in large_clusters:
    cy, cx = region.centroid
    y_int, x_int = int(cy), int(cx)

    # Get the cluster mask
    cluster_mask = missed_labels == region.label

    # Try SAM with the cluster center
    try:
        masks, scores, logits = predictor.predict(
            point_coords=np.array([[x_int, y_int]]),
            point_labels=np.array([1]),
            multimask_output=True
        )
        best_mask = masks[np.argmax(scores)]
        mask_area = best_mask.sum()

        # If SAM returns huge mask, fall back to watershed
        # on the bright cluster itself
        if mask_area > image_area * 0.005:
            raise ValueError("Mask too large, use watershed")

        if 5 < mask_area < 5000:
            mean_int = img[best_mask].mean()
            if mean_int >= sample_mean * 0.5:
                # Watershed to split the SAM mask
                distance  = ndimage.distance_transform_edt(best_mask)
                smooth_d  = ndimage.gaussian_filter(distance, sigma=1.5)
                local_max = morphology.local_maxima(smooth_d)
                markers   = measure.label(local_max)

                if markers.max() >= 2:
                    sub_labels = watershed(-smooth_d, markers,
                                           mask=best_mask)
                    for lbl in range(1, sub_labels.max() + 1):
                        sub = sub_labels == lbl
                        if 5 < sub.sum() < 3000:
                            if img[sub].mean() >= sample_mean * 0.5:
                                masks_current = masks_current | sub
                                pass2_count  += 1
                else:
                    masks_current = masks_current | best_mask
                    pass2_count  += 1

    except Exception:
        # Fallback: watershed directly on the bright cluster
        distance  = ndimage.distance_transform_edt(cluster_mask)
        smooth_d  = ndimage.gaussian_filter(distance, sigma=2)
        local_max = morphology.local_maxima(smooth_d)
        markers   = measure.label(local_max)

        if markers.max() < 1:
            continue

        sub_labels = watershed(-smooth_d, markers,
                               mask=cluster_mask)
        for lbl in range(1, sub_labels.max() + 1):
            sub = sub_labels == lbl
            if 5 < sub.sum() < 3000:
                if img[sub].mean() >= sample_mean * 0.5:
                    masks_current = masks_current | sub
                    pass2_count  += 1

print(f"Pass 2 added: {pass2_count} particles")

# ── Final result ──────────────────────────────────────────────
labels_final = measure.label(masks_current)
print(f"\nBefore rescue : {labels_before.max()} particles")
print(f"After  rescue : {labels_final.max()} particles")
print(f"Gained        : {labels_final.max() - labels_before.max()}")

# Save
np.save('masks_rescued.npy', masks_current)

# ── Visualize ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

overlay = label2rgb(labels_final, image=img,
                    bg_label=0, alpha=0.45)
axes[1].imshow(overlay)
axes[1].set_title(f'After rescue: {labels_final.max()} particles')
axes[1].axis('off')

bnd = np.stack([img] * 3, axis=-1)
bnd[find_boundaries(labels_final, mode='thick')] = [1, 0.1, 0.1]
axes[2].imshow(bnd)
axes[2].set_title('Boundaries')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('sam_rescued.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Check coverage after rescue ───────────────────────────────
still_missed = bright_mask & ~masks_current
coverage     = (bright_mask & masks_current).sum() / bright_mask.sum()
print(f"\nCoverage after rescue: {coverage*100:.1f}%")

# Stats
props = measure.regionprops_table(
    labels_final, intensity_image=img,
    properties=['label', 'centroid', 'area',
                'equivalent_diameter_area',
                'mean_intensity', 'perimeter'])
df = pd.DataFrame(props)
df.columns = ['label', 'cy', 'cx', 'area_px2',
              'diam_px', 'mean_int', 'perim']
df['circularity'] = (4 * np.pi * df['area_px2']
                     / df['perim']**2).round(3)
df_clean = df[df['area_px2'] < 5000].copy()

print(f"Count : {len(df_clean)}")
print(f"Diam  : mean={df_clean['diam_px'].mean():.1f} px")
print(f"Circ  : mean={df_clean['circularity'].mean():.3f}")

df_clean.to_csv('particle_results_rescued.csv', index=False)

# (download handled by maybe_download)
for f in ['masks_rescued.npy', 'sam_rescued.png',
          'particle_results_rescued.csv']:
    if __import__('os').path.exists(f):
        maybe_download(f)
        print(f"Downloaded: {f}")
masks_rescue = masks_current
print('masks_rescue particles:', measure.label(masks_rescue).max())

### 2.5 Box-prompt SAM -> `masks_box`

In [ ]:
# ============================================================
# SAM with box prompts instead of point prompts
# ============================================================
import torch
import gc
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
from skimage import measure
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt

sam    = sam_model_registry["vit_b"](
             checkpoint="sam_vit_b_01ec64.pth")
sam.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
sam.to(device)

predictor   = SamPredictor(sam)
img_for_sam = (img_clahe * 255).astype(np.uint8)
img_rgb     = np.stack([img_for_sam] * 3, axis=-1)
predictor.set_image(img_rgb)

sample_mean = img[sample_mask_norm].mean()
image_area  = img.shape[0] * img.shape[1]
all_masks   = np.zeros(img.shape, dtype=bool)
valid_count = 0

# Box expansion factor: box = sigma * sqrt(2) * expand_factor
# 1.2 = tight box, 1.5 = loose box
EXPAND = 1.0

print(f"Running SAM with BOX prompts on {len(filtered_blobs)} blobs")

for i, blob in enumerate(filtered_blobs):
    y, x, sigma = blob
    y_int, x_int = int(y), int(x)

    # Convert sigma to bounding box
    r = sigma * np.sqrt(2) * EXPAND
    x1 = max(0, x - r)
    y1 = max(0, y - r)
    x2 = min(img.shape[1], x + r)
    y2 = min(img.shape[0], y + r)

    box = np.array([x1, y1, x2, y2])

    try:
        masks, scores, logits = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=box[None, :],          # box prompt
            multimask_output=False     # box prompt → single mask
        )
        best_mask = masks[0]
        mask_area = best_mask.sum()

        if mask_area < 5:
            continue
        if mask_area > image_area * 0.008:
            continue
        if mask_area > (2 * r) ** 2 * 3:  # mask much larger than box
            continue

        mean_int = img[best_mask].mean()
        if mean_int < sample_mean * 0.4:
            continue

        all_masks   = all_masks | best_mask
        valid_count += 1

    except Exception:
        continue

    if (i + 1) % 100 == 0:
        print(f"  [{i+1}/{len(filtered_blobs)}]  valid={valid_count}")

    if (i + 1) % 200 == 0:
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

print(f"\nDone. Valid: {valid_count}")

labels_box = measure.label(all_masks)
print(f"Particles: {labels_box.max()}")

# Quick preview
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

overlay = label2rgb(labels_box, image=img, bg_label=0, alpha=0.45)
axes[1].imshow(overlay)
axes[1].set_title(f'SAM box prompt: {labels_box.max()} particles')
axes[1].axis('off')

bnd = np.stack([img] * 3, axis=-1)
bnd[find_boundaries(labels_box, mode='thick')] = [1, 0.1, 0.1]
axes[2].imshow(bnd)
axes[2].set_title('Boundaries')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('sam_box_result.png', dpi=150, bbox_inches='tight')
plt.show()

masks_box = all_masks.copy()
np.save('masks_box.npy', masks_box)
# (download handled by maybe_download)
maybe_download('masks_box.npy')
maybe_download('sam_box_result.png')

### 2.6 Three-way merge -> `masks_combined`

In [ ]:
masks_combined  = masks_point | masks_rescue | masks_box
labels_point    = measure.label(masks_point)
labels_rescue   = measure.label(masks_rescue)
labels_box_lbl  = measure.label(masks_box)
labels_combined = measure.label(masks_combined)

print(f"Point  : {labels_point.max():>5} particles")
print(f"Rescue : {labels_rescue.max():>5} particles  (incremental on top of point)")
print(f"Box    : {labels_box_lbl.max():>5} particles")
print(f"Merged : {labels_combined.max():>5} particles")

np.save('masks_combined.npy',  masks_combined)
np.save('labels_combined.npy', labels_combined)
maybe_download('masks_combined.npy')

### 2.7 Final visualization (fill / boundary / solid)

In [ ]:
# Recompute combined mask (idempotent)
masks_combined  = masks_point | masks_rescue | masks_box
labels_combined = measure.label(masks_combined)
print(f"Combined: {labels_combined.max()} particles")

# Variant A: semi-transparent single-color fill
img_norm = (img - img.min()) / (img.max() - img.min())
rgb = np.stack([img_norm] * 3, axis=-1).copy()
color = np.array(mcolors.to_rgb('magenta'))   # change color here
alpha = 0.4
rgb[labels_combined > 0] = ((1 - alpha) * rgb[labels_combined > 0] + alpha * color)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)
ax.set_title(f'Combined: {labels_combined.max()} particles', fontsize=14)
ax.axis('off')
plt.tight_layout(); plt.savefig('combined_final.png', dpi=150, bbox_inches='tight'); plt.show()

# Variant B: colored boundaries only, particle interior keeps grayscale
img_norm = (img - img.min()) / (img.max() - img.min())
rgb = np.stack([img_norm] * 3, axis=-1).copy()
boundaries = find_boundaries(labels_combined, mode='thick')
rgb[boundaries] = mcolors.to_rgb('gold')      # change color here
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)
ax.set_title(f'{labels_combined.max()} particles', fontsize=14)
ax.axis('off')
plt.tight_layout(); plt.savefig('final.png', dpi=150, bbox_inches='tight'); plt.show()

# Variant C: solid opaque fill
img_norm = (img - img.min()) / (img.max() - img.min())
rgb = np.stack([img_norm] * 3, axis=-1).copy()
color = np.array(mcolors.to_rgb('deepskyblue'))
rgb[labels_combined > 0] = color
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)
ax.set_title(f'{labels_combined.max()} particles', fontsize=14)
ax.axis('off')
plt.tight_layout(); plt.savefig('final_solid.png', dpi=150, bbox_inches='tight'); plt.show()

maybe_download('combined_final.png')
maybe_download('final.png')
maybe_download('final_solid.png')

### 2.8 Final statistics & export

In [ ]:
props = measure.regionprops_table(
    labels_combined, intensity_image=img,
    properties=['label', 'centroid', 'area', 'equivalent_diameter_area',
                'mean_intensity', 'max_intensity', 'perimeter'])
df = pd.DataFrame(props)
df.columns = ['label', 'cy', 'cx', 'area_px2', 'diam_px',
              'mean_int', 'max_int', 'perim']
df['circularity'] = (4 * np.pi * df['area_px2'] / df['perim']**2).round(3)
df_final = df[df['area_px2'] < 5000].copy()

print(f"Final particle count : {len(df_final)}")
print(f"  Equiv diameter : mean={df_final['diam_px'].mean():.1f}px, "
      f"std={df_final['diam_px'].std():.1f}px")
print(f"  Circularity    : mean={df_final['circularity'].mean():.3f}")
df_final.to_csv('particle_results_final.csv', index=False)
maybe_download('particle_results_final.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df_final['diam_px'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Equivalent Diameter (px)'); axes[0].set_ylabel('Count'); axes[0].set_title('Size')
axes[1].hist(df_final['mean_int'], bins=30, color='orangered', edgecolor='white')
axes[1].set_xlabel('Mean Intensity'); axes[1].set_ylabel('Count'); axes[1].set_title('Intensity')
axes[2].hist(df_final['circularity'], bins=30, color='seagreen', edgecolor='white')
axes[2].set_xlabel('Circularity'); axes[2].set_ylabel('Count'); axes[2].set_title('Circularity')
plt.tight_layout(); plt.savefig('stats_final.png', dpi=150, bbox_inches='tight'); plt.show()
maybe_download('stats_final.png')